# EC424: GIS สำหรับเศรษฐศาสตร์

**เป้าหมาย:** ใช้ภาพถ่ายดาวเทียมตอบคำถามเชิงเศรษฐศาสตร์: การขยายตัวของเมือง การใช้ที่ดิน และการคาดคะเนอนาคต

**Case study:** กรุงเทพฯ + ปริมณฑล (ภาคกลางของไทย) · เปรียบเทียบ LULC (Land Use / Land Cover) ปี 2015 vs 2025 + จำลองอนาคต

---

### แผนงาน

| # | หัวข้อ | ผลลัพธ์ |
|---|-------|--------|
| 1 | Setup + libraries | environment พร้อม |
| 2 | ค้นหาภาพ Landsat | 2 scenes (2015 + 2025) |
| 3 | ดูภาพ preview | เห็นภูมิทัศน์ที่เปลี่ยน |
| 4 | Spectral signatures | รู้จัก fingerprint ของวัตถุ |
| 5 | NDVI / NDWI / NDBI | ดัชนีพืช / น้ำ / เมือง |
| 6 | Random Forest | LULC map 5 classes (pseudo-labeled) |
| 7 | Inference ทั้งพื้นที่ | map ทั้งสองช่วงเวลา |
| 8 | Area change | ตัวเลขการเปลี่ยนแปลง |
| 9 | CA-Markov | จำลองอนาคต |
| 10 | Econ interpretation | applications + Capstone |

**รันบน:** Google Colab (CPU-only OK) · **เวลา:** ~5-10 นาที · **ไม่ต้อง login อะไร**

## 🔧 1. ติดตั้ง Libraries + Imports

**Libraries ที่ใช้:**

| Library | บทบาท |
|---------|-------|
| `pystac_client` | ค้นหาภาพจาก STAC catalog |
| `planetary_computer` | sign URL ให้เข้าถึงได้ |
| `rasterio` | อ่าน/เขียนภาพ raster (GeoTIFF) |
| `geopandas` | จัดการข้อมูล vector (points/polygons) |
| `scikit-learn` | Random Forest, scaler, metrics |
| `tqdm` | progress bar |


In [ ]:
!pip install -q pystac_client planetary_computer rasterio geopandas tqdm


In [ ]:
# ===== Geospatial =====
import pystac_client
import planetary_computer
import rasterio
from rasterio.enums import Resampling
from rasterio.mask import mask as rio_mask
from pyproj import Transformer
import geopandas as gpd
from shapely.geometry import box

# ===== Data / ML =====
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

# ===== Visualization =====
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.patches import Patch

# ===== Utilities =====
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility: seed ทั้ง NumPy และ Python
np.random.seed(42)

# Consistent plotting style
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('✓ Libraries ready')

## 🛰️ 2. ค้นหาภาพดาวเทียม Landsat

เราใช้ **Microsoft Planetary Computer** ซึ่งเป็น cloud catalog ของ NASA/USGS Landsat
ที่ public ทั้งหมด (ไม่ต้องสมัคร account)

**Workflow:**
1. เปิด STAC catalog (SpatioTemporal Asset Catalog: มาตรฐาน metadata ภาพดาวเทียม)
2. Query ด้วย bounding box + ช่วงวันที่ + cloud cover
3. ได้ list ของ scenes ที่ตรงเงื่อนไข

> **AOI (Area of Interest):** กรุงเทพฯ + ปริมณฑล (~75 × 90 km²) · รวม BKK, Nonthaburi, Pathum Thani, Ayutthaya บางส่วน<br>
> **ช่วงเวลา:** 2015 (past) vs 2025 (present): ห่างกัน 10 ปี

In [ ]:
# ============ Configuration ============
STAC_URL      = 'https://planetarycomputer.microsoft.com/api/stac/v1'
COLLECTION    = 'landsat-c2-l2'    # Landsat Collection 2 Level 2 (Surface Reflectance)

# 🇹🇭 Bangkok + ปริมณฑล (ภาคกลางไทย)
# [min_lon, min_lat, max_lon, max_lat]
AOI_BBOX      = [100.3, 13.45, 100.9, 14.25]

MAX_CLOUD_PCT = 10                  # % cloud cover threshold (ต้องการภาพที่เมฆน้อยเพื่อความแม่นยำ)


def search_landsat(bbox, date_range, max_cloud=MAX_CLOUD_PCT):
    """Search Landsat scenes intersecting bbox in a date range."""
    catalog = pystac_client.Client.open(
        STAC_URL,
        modifier=planetary_computer.sign_inplace,
    )
    search = catalog.search(
        collections=[COLLECTION],
        bbox=bbox,
        datetime=date_range,
        query={'eo:cloud_cover': {'lt': max_cloud}},
    )
    return search.item_collection()


items_2015 = search_landsat(AOI_BBOX, '2015-01-01/2015-12-31')
items_2025 = search_landsat(AOI_BBOX, '2025-01-01/2025-12-31')

print(f'🛰️  Found {len(items_2015):>2} scenes in 2015 (cloud < {MAX_CLOUD_PCT}%)')
print(f'🛰️  Found {len(items_2025):>2} scenes in 2025 (cloud < {MAX_CLOUD_PCT}%)')

### เลือก scene ที่คุณภาพดีที่สุด

Scene แต่ละอันมี cloud cover, ฤดูกาล, angle ต่างกัน: เราเลือกที่เมฆน้อย
และเป็นฤดูกาลเดียวกัน (ให้เทียบกันได้ยุติธรรม)


In [ ]:
def list_scenes(items, label):
    """Print table of available scenes to help picking."""
    rows = []
    for i, item in enumerate(items):
        rows.append({
            'idx': i,
            'date': item.datetime.date(),
            'cloud_%': round(item.properties['eo:cloud_cover'], 1),
            'platform': item.properties.get('platform', ''),
        })
    df = pd.DataFrame(rows)
    print(f'\n=== {label} ===')
    print(df.to_string(index=False))
    return df

_ = list_scenes(items_2015, '2015 scenes')
_ = list_scenes(items_2025, '2025 scenes')


In [ ]:
from shapely.geometry import shape


def _list_valid_scenes(items, bbox, min_coverage):
    """Filter scenes ที่ cover bbox ≥ min_coverage."""
    bbox_geom = box(*bbox)
    result = []
    for i, item in enumerate(items):
        footprint = shape(item.geometry)
        if not footprint.intersects(bbox_geom):
            continue
        coverage = footprint.intersection(bbox_geom).area / bbox_geom.area
        if coverage < min_coverage:
            continue
        result.append({
            'idx': i,
            'item': item,
            'coverage': coverage,
            'cloud': item.properties.get('eo:cloud_cover', 100),
            'month': item.datetime.month,
            'date': item.datetime.date(),
        })
    return result


def find_best_scene_pair(items_a, items_b, bbox, min_coverage=0.75, month_tolerance=1):
    """เลือก scene pair (a, b) ให้เดือนใกล้กัน + cloud รวมต่ำ.

    Args:
        items_a, items_b: STAC items ของ 2 ช่วงเวลา
        bbox: AOI bbox [min_lon, min_lat, max_lon, max_lat]
        min_coverage: coverage ขั้นต่ำ (0.75 = cover ≥ 75% ของ bbox)
        month_tolerance: เดือนห่างกันได้ ± ไม่เกิน (1 = ±1 เดือน)

    Returns:
        (scene_a, scene_b, month_diff): แต่ละ scene เป็น dict ที่มี 'idx', 'item', ...
    """
    valid_a = _list_valid_scenes(items_a, bbox, min_coverage)
    valid_b = _list_valid_scenes(items_b, bbox, min_coverage)

    if not valid_a or not valid_b:
        raise RuntimeError(
            f'ไม่มี scene ที่ cover ≥ {min_coverage*100:.0f}%: '
            f'ลอง min_coverage ต่ำลง หรือเพิ่ม MAX_CLOUD_PCT'
        )

    # หา pair ที่เดือนใกล้กัน (circular distance เผื่อ Dec vs Jan)
    candidates = []
    for a in valid_a:
        for b in valid_b:
            diff = abs(a['month'] - b['month'])
            month_diff = min(diff, 12 - diff)  # circular: Dec-Jan = 1
            if month_diff <= month_tolerance:
                # score: cloud รวม + penalty เดือนห่าง
                score = a['cloud'] + b['cloud'] + month_diff * 5
                candidates.append((a, b, month_diff, score))

    if not candidates:
        raise RuntimeError(
            f'ไม่มี pair ที่เดือนห่างกัน ≤ ±{month_tolerance}: '
            'ลองขยาย month_tolerance หรือเพิ่ม MAX_CLOUD_PCT'
        )

    # เลือก pair ที่ score น้อยสุด (cloud ต่ำรวม + เดือนใกล้กัน)
    candidates.sort(key=lambda x: x[3])
    return candidates[0][:3]


# ===== เลือก scene pair =====
scene_a, scene_b, month_diff = find_best_scene_pair(
    items_2015, items_2025, AOI_BBOX,
    min_coverage=0.75, month_tolerance=1,
)

idx_2015, item_2015 = scene_a['idx'], scene_a['item']
idx_2025, item_2025 = scene_b['idx'], scene_b['item']

print(f'✓ 2015: idx={idx_2015} · {scene_a["date"]} '
      f'· cloud {scene_a["cloud"]:.1f}% · coverage {scene_a["coverage"]*100:.0f}%')
print(f'✓ 2025: idx={idx_2025} · {scene_b["date"]} '
      f'· cloud {scene_b["cloud"]:.1f}% · coverage {scene_b["coverage"]*100:.0f}%')
print(f'📅 Month diff: {month_diff} เดือน (เทียบกันได้ยุติธรรม)')

## 🖼️ 3. ดูภาพ Preview (True Color)

เริ่มจากภาพ **RGB true color** (ตามองเห็น) ก่อน เพื่อให้ได้ intuition ของพื้นที่ที่เรากำลังวิเคราะห์
Planetary Computer ได้ pre-rendered ไว้ให้แล้ว (asset ชื่อ `rendered_preview`)


In [ ]:
def read_preview(item):
    """Read pre-rendered RGB preview image from STAC item."""
    with rasterio.open(item.assets['rendered_preview'].href) as src:
        # rasterio: (C, H, W) → matplotlib: (H, W, C)
        return src.read().transpose(1, 2, 0)


fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, item, year in zip(axes, [item_2015, item_2025], [2015, 2025]):
    ax.imshow(read_preview(item))
    ax.set_title(f'Bangkok {year} · {item.datetime.date()}', fontsize=13, fontweight='bold')
    ax.axis('off')
fig.suptitle('Landsat True-Color Preview: 2015 vs 2025', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 📊 4. Spectral Signatures: "ลายนิ้วมือ" ของวัตถุ

### Landsat บันทึก 6 ย่านคลื่นที่เราจะใช้

| Band | ช่วงคลื่น (μm) | เห็นอะไรชัด |
|------|--------------|------------|
| **Blue** (B2) | 0.45-0.51 | น้ำ, อากาศ |
| **Green** (B3) | 0.53-0.59 | พืชที่สุขภาพดี |
| **Red** (B4) | 0.64-0.67 | ใบไม้ดูดซับ (chlorophyll) |
| **NIR** (B5) | 0.85-0.88 | พืชสะท้อนมาก |
| **SWIR-1** (B6) | 1.57-1.65 | ความชื้นในพืช/ดิน |
| **SWIR-2** (B7) | 2.11-2.29 | วัตถุก่อสร้าง, ดินแห้ง |

### DN → Reflectance

ค่าใน Landsat ไม่ใช่ reflectance ตรง ๆ: เป็น **DN (Digital Number)** ที่ถูก scale<br>
สูตร (Landsat Collection 2 Level 2): **`reflectance = DN × 0.0000275 - 0.2`**<br>
(reflectance อยู่ในช่วง 0-1 = 0% ถึง 100%)


In [ ]:
# 6 bands ที่จะใช้ตลอด notebook
BANDS = ['blue', 'green', 'red', 'nir08', 'swir16', 'swir22']

# Central wavelength ของแต่ละ band (μm): ใช้ plot
BAND_WAVELENGTH = {
    'blue':   0.48, 'green': 0.56, 'red':    0.66,
    'nir08':  0.87, 'swir16': 1.61, 'swir22': 2.20,
}


def dn_to_reflectance(dn):
    """Convert Landsat Collection 2 L2 DN to surface reflectance [0, 1]."""
    return dn * 0.0000275 - 0.2


def sample_spectral(item, lon, lat, bands=BANDS, window=3):
    """Sample Landsat bands at a (lon, lat) point with N×N window mean.

    ใช้ค่าเฉลี่ย window 3×3 (90m × 90m) แทน single pixel
    → ทนต่อ mixed pixel / misregistration / cloud edge
    """
    half = window // 2
    values = []
    for band in bands:
        with rasterio.open(item.assets[band].href) as src:
            transformer = Transformer.from_crs(
                'EPSG:4326', src.crs, always_xy=True
            )
            x, y = transformer.transform(lon, lat)
            row, col = src.index(x, y)
            arr = src.read(
                1,
                window=((row - half, row + half + 1),
                        (col - half, col + half + 1)),
            )
            valid = arr[arr > 0]  # ตัด nodata (DN=0)
            mean_dn = float(valid.mean()) if valid.size else 0.0
            values.append(dn_to_reflectance(mean_dn))
    return values


# Reference points ที่รู้แน่ว่าเป็น land cover อะไร (กรุงเทพฯ + ปริมณฑล)
REF_POINTS = {
    'Forest':   (100.5472, 13.6833),   # Bang Krachao "green lung"
    'Urban':    (100.5340, 13.7450),   # Siam / Silom CBD
    'Water':    (100.5060, 13.7300),   # Chao Phraya กลางเมือง (สะพานพระราม 8)
    'Cropland': (100.3800, 13.9100),   # Rice fields Bang Bua Thong / Pathum Thani
}

print('📍 Sampling spectral signatures at 4 reference points...')
signatures = {
    label: sample_spectral(item_2025, lon, lat)
    for label, (lon, lat) in REF_POINTS.items()
}
print('✓ Done')

In [ ]:
# Plot: ใช้ wavelength จริง (μm) เป็นแกน X แทนชื่อ band
PLOT_COLORS = {'Forest':'#2e7d32', 'Urban':'#616161',
               'Water':'#1976d2', 'Cropland':'#f57c00'}
PLOT_MARKERS = {'Forest':'o', 'Urban':'s', 'Water':'^', 'Cropland':'D'}

wavelengths = [BAND_WAVELENGTH[b] for b in BANDS]

fig, ax = plt.subplots(figsize=(10, 5.5))
for label, values in signatures.items():
    ax.plot(wavelengths, values,
            marker=PLOT_MARKERS[label], markersize=8,
            label=label, color=PLOT_COLORS[label],
            linewidth=2.2, alpha=0.85)

# shade visible range + NIR/SWIR ranges
ax.axvspan(0.4, 0.7, alpha=0.08, color='yellow', label='_visible')
ax.axvspan(0.7, 1.0, alpha=0.05, color='red', label='_nir')
ax.axvspan(1.0, 2.5, alpha=0.04, color='gray', label='_swir')

ax.set_xlabel('Wavelength (μm)', fontsize=12)
ax.set_ylabel('Surface Reflectance', fontsize=12)
ax.set_title('Spectral Signatures: วัตถุแต่ละประเภทมี "fingerprint" เฉพาะ',
             fontsize=13, fontweight='bold')
ax.set_ylim(-0.05, None)
ax.legend(loc='upper right', frameon=True)
ax.grid(alpha=0.3)

# annotate visible region
ax.text(0.55, ax.get_ylim()[1]*0.95, 'Visible', ha='center',
        fontsize=9, style='italic', color='#666')
ax.text(0.87, ax.get_ylim()[1]*0.95, 'NIR', ha='center',
        fontsize=9, style='italic', color='#666')
ax.text(1.9, ax.get_ylim()[1]*0.95, 'SWIR', ha='center',
        fontsize=9, style='italic', color='#666')

plt.tight_layout()
plt.show()


### 🤔 สังเกตอะไรได้จากกราฟ?

- **🌲 Forest:** Red ต่ำ (chlorophyll ดูดซับ) → NIR พุ่งสูง (สะท้อนเพื่อไม่ให้ร้อน)
- **🏙️ Urban:** reflectance ค่อนข้างสูงทุกย่าน โดยเฉพาะ SWIR (วัตถุก่อสร้าง)
- **💧 Water:** Blue สูง → NIR/SWIR ต่ำมาก (น้ำดูดซับ IR)
- **🌾 Cropland:** คล้ายพืชแต่ต่ำกว่า (ขึ้นกับฤดูกาล)

→ ความต่างนี้คือ **กุญแจ** ที่ทำให้ Random Forest แยก LULC ได้ในขั้นถัดไป


## 🌿 5. Spectral Indices: NDVI / NDWI / NDBI

แทนที่จะใช้ทั้ง 6 bands เราสามารถสร้าง **index** ที่เน้นเฉพาะสิ่งที่สนใจ ด้วยการเอา 2 bands มา normalize

| Index | สูตร | เน้นอะไร | Threshold |
|-------|------|--------|----------|
| **NDVI** | (NIR − Red) / (NIR + Red) | ☘️ Vegetation | > 0.3 = พืชสุขภาพดี |
| **NDWI** (McFeeters 1996) | (Green − NIR) / (Green + NIR) | 💧 Water | > 0 = แหล่งน้ำ |
| **NDBI** | (SWIR − NIR) / (SWIR + NIR) | 🏙️ Built-up | > -0.05 = อาคาร/ถนน |

> **ทำไมต้อง normalize?** ความสว่างสัมบูรณ์ขึ้นกับแดด/มุมถ่าย → เทียบข้าม scene ไม่ได้<br>
> Normalize → ค่า -1 ถึง +1 → เทียบได้ทุก scene


In [ ]:
def read_band_array(item, band_name, resampling=Resampling.nearest):
    """Read one band as float32 array (nodata → NaN) + profile."""
    with rasterio.open(item.assets[band_name].href) as src:
        arr = src.read(1, resampling=resampling).astype('float32')
        profile = src.profile
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)
    return arr, profile


# Read 4 bands needed for indices (from 2025 scene)
print('📥 Reading bands for index computation...')
red,   _ = read_band_array(item_2025, 'red')
green, _ = read_band_array(item_2025, 'green')
nir,   _ = read_band_array(item_2025, 'nir08')
swir,  _ = read_band_array(item_2025, 'swir22')

# Compute indices (eps กัน divide by zero)
EPS = 1e-10
ndvi = (nir   - red)  / (nir   + red  + EPS)
ndwi = (green - nir)  / (green + nir  + EPS)
ndbi = (swir  - nir)  / (swir  + nir  + EPS)

# Summary table
summary = pd.DataFrame({
    'Index': ['NDVI', 'NDWI', 'NDBI'],
    'Min':   [np.nanmin(ndvi), np.nanmin(ndwi), np.nanmin(ndbi)],
    'Max':   [np.nanmax(ndvi), np.nanmax(ndwi), np.nanmax(ndbi)],
    'Mean':  [np.nanmean(ndvi), np.nanmean(ndwi), np.nanmean(ndbi)],
}).round(2)
print(summary.to_string(index=False))


In [ ]:
# Visualize 3 indices side-by-side with consistent colormaps
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

plots = [
    (ndvi, 'NDVI (Vegetation)', 'Greens', -0.2, 0.8),
    (ndwi, 'NDWI (Water)',      'Blues',  -0.5, 0.5),
    (ndbi, 'NDBI (Built-up)',   'Reds',   -0.3, 0.3),
]

for ax, (data, title, cmap, vmin, vmax) in zip(axes, plots):
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)

fig.suptitle('Remote Sensing Indices: 2025', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


### ประเด็นที่ต้องรู้

1. **ค่า NaN (สีโปร่งใส)** = nodata หรือพื้นที่นอก swath ของดาวเทียม
2. **Threshold ไม่ใช่กฎเหล็ก**: NDVI > 0.3 เป็น rule of thumb ขึ้นกับพื้นที่และฤดูกาล
3. **Index เดียวแยกไม่ละเอียด**: เราจึงใช้ ML ในขั้นถัดไปเพื่อแยก 8 classes


## 🤖 6. LULC Classification ด้วย Random Forest

### วิธี Pseudo-labeling (priority-based)

ไม่มี training points ของไทยที่ label ไว้แล้ว: เราใช้ **spectral indices** เป็น proxy label
โดยใช้ **ลำดับความสำคัญ**: water → veg → crop → built-up → bareland
(pixel ที่ตรงหลายเงื่อนไข เลือกตามลำดับก่อน → ไม่ซ้อนกัน)

```
1. NDWI > 0.1                          → Water body     (น้ำ)
2. NDVI > 0.5                          → Vegetation     (ป่า/ต้นไม้ดก)
3. NDVI 0.25-0.5 & NDBI < 0            → Cropland       (นาข้าว/พืชเกษตร)
4. NDBI > -0.05 & NDVI < 0.3           → Built-up       (เมือง: relaxed สำหรับเขตร้อน)
5. NDVI < 0.05 & NDBI < -0.1           → Bareland       (ที่ว่างจริง: strict)
```

> **ทำไม threshold แบบนี้?**
> เมืองไทยมี "เมืองแบบผสม": มีต้นไม้แซม → NDBI ไม่สูงมาก<br>
> **Bareland** แท้ ๆ หายากในเมือง → threshold ต้องเข้ม ไม่ให้ built-up หลุดไปเป็น bareland

### 5 Classes (simplified สำหรับ Thai LULC)

| ID | Class | สี |
|----|-------|----|
| 10 | Vegetation | 🟢 dark green |
| 40 | Cropland | 🟡 yellow |
| 50 | Built-up | 🔴 red |
| 60 | Bareland | ⚪ light grey |
| 80 | Water body | 🔵 blue |

In [ ]:
# ============ LULC class definitions ============
LULC_NAMES = {
    0:  'Unclassified',
    10: 'Vegetation',
    20: 'Shrubland',
    30: 'Grassland',
    40: 'Cropland',
    50: 'Built-up',
    60: 'Bareland',
    80: 'Water body',
}
LULC_COLORS_LIST = [
    '#000000',  # Unclassified
    '#1b5e20',  # Vegetation (dark green)
    '#fb8c00',  # Shrubland (orange)
    '#9ccc65',  # Grassland (light green)
    '#fdd835',  # Cropland (yellow)
    '#e53935',  # Built-up (red)
    '#bdbdbd',  # Bareland (light grey)
    '#1976d2',  # Water body (blue)
]

# Colormap for visualization
lulc_cmap = colors.ListedColormap(LULC_COLORS_LIST)
lulc_bounds = list(LULC_NAMES.keys()) + [100]
lulc_norm = colors.BoundaryNorm(lulc_bounds, lulc_cmap.N)

# ============ Priority-based Pseudo-labeling ============
# ลำดับ: water → vegetation → cropland → built-up → bareland
# pixel ที่ match หลายเงื่อนไข → เลือกตาม priority (ไม่ overlap)
is_water    = (ndwi > 0.1)
is_veg      = (ndvi > 0.5) & ~is_water
is_crop     = (ndvi > 0.25) & (ndvi <= 0.5) & (ndbi < 0) & ~is_water & ~is_veg
# Built-up: relaxed สำหรับเขตร้อน (BKK มีต้นไม้แซม → NDBI ไม่สูงมาก)
is_builtup  = (ndbi > -0.05) & (ndvi < 0.3) & ~is_water & ~is_veg & ~is_crop
# Bareland (strict): ต้องทั้ง NDVI ต่ำมาก AND NDBI ต่ำ (ไม่เหลือ builtup)
is_bareland = (ndvi < 0.05) & (ndbi < -0.1) & ~is_water & ~is_veg & ~is_crop & ~is_builtup

LABEL_MASKS = {
    10: is_veg,
    40: is_crop,
    50: is_builtup,
    60: is_bareland,
    80: is_water,
}

print('🔬 Pseudo-label candidates per class:')
for class_id, mask in LABEL_MASKS.items():
    valid = int(np.nansum(mask))
    print(f'   {class_id:>2} ({LULC_NAMES[class_id]:<12}): {valid:>8,} pixels')

# Quick sanity check: total covered vs total valid pixels
total_labeled = sum(int(np.nansum(m)) for m in LABEL_MASKS.values())
total_valid = int(np.sum(~np.isnan(ndvi)))
print(f'\n   Total labeled: {total_labeled:,} / {total_valid:,} valid pixels '
      f'({total_labeled/total_valid*100:.1f}%)')

In [ ]:
# ============ Sample pixels จาก masks + ดึง 6 band features ============
print('📥 Loading 6 bands from 2025 scene (ใช้ทำ features)...')
band_arrays = {}
for b in BANDS:
    arr, _ = read_band_array(item_2025, b)
    band_arrays[b] = arr

# Sample N pixels ต่อ class
N_PER_CLASS = 200
rng = np.random.default_rng(seed=42)

X_list, y_list = [], []
print('\n🎯 Sampling training pixels per class:')
for class_id, mask in LABEL_MASKS.items():
    valid_idx = np.where(mask)
    n_valid = len(valid_idx[0])
    if n_valid == 0:
        print(f'   ⚠️  class {class_id} ({LULC_NAMES[class_id]}): no pixels: skipped')
        continue
    n_sample = min(N_PER_CLASS, n_valid)
    picks = rng.choice(n_valid, n_sample, replace=False)

    n_added = 0
    for p in picks:
        r, c = valid_idx[0][p], valid_idx[1][p]
        features = [band_arrays[b][r, c] for b in BANDS]
        # skip NaN / nodata
        if any(np.isnan(f) or f == 0 for f in features):
            continue
        X_list.append(features)
        y_list.append(class_id)
        n_added += 1
    print(f'   {class_id:>2} ({LULC_NAMES[class_id]:<12}): {n_added}/{n_sample}')

X_raw = np.array(X_list, dtype='float32')
y_raw = np.array(y_list, dtype='int32')
print(f'\n✓ Total training samples: {len(X_raw)}')

In [ ]:
# ============ Prepare X, y ============
X = X_raw
y = y_raw

# เพิ่ม Unclassified class (all-zero) ~50 samples
# เพื่อให้โมเดลรู้จัก pixel ที่ไม่มีข้อมูล (edge)
N_UNCLASS = 50
X = np.vstack([X, np.zeros((N_UNCLASS, len(BANDS)))])
y = np.concatenate([y, np.zeros(N_UNCLASS, dtype=int)])

# Scale to [0, 1]
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, stratify=y, random_state=42,
)
print(f'📊 Train: {X_train.shape[0]} samples · Test: {X_test.shape[0]} samples')
print(f'   Classes: {sorted(np.unique(y))}')

In [ ]:
# ============ Train Random Forest ============
model = RandomForestClassifier(
    n_estimators=100,   # จำนวน trees
    max_depth=20,       # ลึกสุด (มากไป = overfit)
    random_state=42,
    n_jobs=-1,          # ใช้ CPU ทุก core
)
model.fit(X_train, y_train)

# ============ Evaluate ============
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'✓ Test accuracy: {acc:.2%}')

# Confusion matrix
labels_num = sorted(np.unique(y_test))
labels_str = [LULC_NAMES[l] for l in labels_num]
cm = confusion_matrix(y_test, y_pred, labels=labels_num)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# [1] Confusion matrix heatmap
ax1 = axes[0]
ConfusionMatrixDisplay(cm, display_labels=labels_str).plot(
    ax=ax1, cmap='Blues', xticks_rotation=45, colorbar=False
)
ax1.set_title(f'Confusion Matrix (accuracy = {acc:.2%})', fontsize=12, fontweight='bold')
ax1.grid(False)

# [2] Feature importance: ดู band ไหนสำคัญสุด
ax2 = axes[1]
importances = model.feature_importances_
ax2.barh(BANDS, importances, color='#1976d2')
ax2.set_xlabel('Feature Importance')
ax2.set_title('ความสำคัญของแต่ละ band', fontsize=12, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


## 🗺️ 7. Inference: LULC Map ของทั้ง AOI

**ขั้นตอน:**
1. โหลด 6 bands ของทั้ง scene (crop ด้วย AOI polygon)
2. Downsample 4× เพื่อความเร็ว (สำหรับ demo)
3. Reshape เป็น (H·W, 6), scale, predict
4. Reshape กลับเป็น (H, W) → LULC map

> **Downsample:** เอาทุก 4 pixel → ลดขนาดภาพลง 16 เท่า (ได้ 1/16 ของ pixel)<br>
> งานจริงไม่ downsample: ใช้ full resolution 30m


In [ ]:
def build_datacube(item, bands=BANDS, aoi_bbox=None, downsample=1):
    """Build a (H, W, C) datacube, optionally cropped to AOI bbox + downsampled.

    ใช้ bbox (ไม่ใช่ polygon) เพราะง่ายและเร็วกว่า
    """
    arrays = []
    profile = None

    # Build a shapely box for cropping (in EPSG:4326)
    bbox_geom = box(*aoi_bbox) if aoi_bbox is not None else None

    for band in bands:
        with rasterio.open(item.assets[band].href) as src:
            if bbox_geom is not None:
                # Reproject bbox to raster CRS
                bbox_reproj = (gpd.GeoSeries([bbox_geom], crs='EPSG:4326')
                               .to_crs(src.crs))
                arr, transform = rio_mask(src, bbox_reproj.geometry.tolist(), crop=True)
                arr = arr[0]
            else:
                arr = src.read(1)
                transform = src.transform

            if profile is None:
                profile = src.profile.copy()
                profile.update(transform=transform,
                               height=arr.shape[0],
                               width=arr.shape[1])
            arrays.append(arr)

    cube = np.stack(arrays, axis=-1)  # (H, W, C)

    if downsample > 1:
        cube = cube[::downsample, ::downsample, :]
        profile['height'] = cube.shape[0]
        profile['width']  = cube.shape[1]

    return cube, profile


DOWNSAMPLE = 4  # 4 → ได้ภาพเร็วขึ้น 16x

print('📥 Building datacubes (crop ด้วย bbox + downsample)...')
cube_2025, prof_2025 = build_datacube(item_2025, aoi_bbox=AOI_BBOX, downsample=DOWNSAMPLE)
cube_2015, prof_2015 = build_datacube(item_2015, aoi_bbox=AOI_BBOX, downsample=DOWNSAMPLE)
print(f'   2015 cube: {cube_2015.shape}')
print(f'   2025 cube: {cube_2025.shape}')

In [ ]:
def predict_lulc(cube, model, scaler):
    """Apply trained model to every pixel of the cube.

    cube shape: (H, W, C) → returns (H, W) class id map
    """
    h, w, c = cube.shape
    flat = cube.reshape(-1, c)
    flat_scaled = scaler.transform(flat)
    pred_flat = model.predict(flat_scaled)
    return pred_flat.reshape(h, w)


print('🔮 Predicting LULC maps...')
lulc_2015 = predict_lulc(cube_2015, model, scaler)
lulc_2025 = predict_lulc(cube_2025, model, scaler)
print('✓ Done')


In [ ]:
# Visualize side-by-side with shared legend
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, lulc, year in zip(axes, [lulc_2015, lulc_2025], [2015, 2025]):
    ax.imshow(lulc, cmap=lulc_cmap, norm=lulc_norm, interpolation='nearest')
    ax.set_title(f'LULC {year}', fontsize=13, fontweight='bold')
    ax.axis('off')

# Shared legend below
legend_patches = [
    Patch(facecolor=LULC_COLORS_LIST[i], label=name)
    for i, name in enumerate(LULC_NAMES.values())
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4,
           fontsize=10, bbox_to_anchor=(0.5, -0.08), frameon=False)

fig.suptitle('Land Use / Land Cover: 2015 vs 2025', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 📐 8. วัดการเปลี่ยนแปลงพื้นที่ (2015 → 2025)

Landsat pixel size = **30m × 30m = 900 m²** (~0.0009 km²)<br>
เมื่อเรา downsample 4× 1 pixel แทนพื้นที่ 120m × 120m = 14,400 m² = **900 × 16 m²**


In [ ]:
def compute_area_by_class(lulc_map, pixel_area_m2):
    """Count pixels per class and convert to km²."""
    unique, counts = np.unique(lulc_map, return_counts=True)
    df = pd.DataFrame({
        'class_id':   unique,
        'pixels':     counts,
        'area_km2':   counts * pixel_area_m2 / 1_000_000,
    })
    df['class_name'] = df['class_id'].map(LULC_NAMES)
    return df.set_index('class_name')[['area_km2']]


# ปรับ pixel area ตาม downsample factor
PIXEL_AREA_M2 = 900 * (DOWNSAMPLE ** 2)

area_2015 = compute_area_by_class(lulc_2015, PIXEL_AREA_M2)
area_2025 = compute_area_by_class(lulc_2025, PIXEL_AREA_M2)

# Combine + compute change
area_df = pd.concat([area_2015, area_2025], axis=1).fillna(0)
area_df.columns = ['2015 (km²)', '2025 (km²)']
area_df['Δ km²']     = (area_df['2025 (km²)'] - area_df['2015 (km²)']).round(2)
area_df['Change %']  = ((area_df['2025 (km²)'] - area_df['2015 (km²)'])
                        / area_df['2015 (km²)'].replace(0, np.nan) * 100).round(1)
area_df = area_df.sort_values('Change %', ascending=False, na_position='last')
area_df.round(2)


In [ ]:
# Visualize change: 2 subplots: absolute area + % change
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# [1] Absolute area: stacked bar
ax1 = axes[0]
area_df[['2015 (km²)', '2025 (km²)']].plot.barh(
    ax=ax1, color=['#90a4ae', '#1976d2'], width=0.7,
)
ax1.set_xlabel('Area (km²)')
ax1.set_title('ขนาดพื้นที่แต่ละ class', fontsize=12, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
ax1.legend(frameon=False)

# [2] Change %: colored bars
ax2 = axes[1]
chg = area_df['Change %'].dropna()
bar_colors = ['#2e7d32' if v < 0 else '#c62828' for v in chg]
chg.plot.barh(ax=ax2, color=bar_colors)
ax2.set_xlabel('Change (%)')
ax2.set_title('การเปลี่ยนแปลง 2015 → 2025', fontsize=12, fontweight='bold')
ax2.axvline(0, color='black', linewidth=0.5)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


## 🔮 9. จำลองอนาคต: Simplified CA-Markov

**Cellular Automata (CA) + Markov Chain** = วิธีจำลองการเปลี่ยนแปลงของแต่ละ pixel ในอนาคต

| ส่วนประกอบ | ทำหน้าที่ |
|-----------|---------|
| **Markov Chain** | เรียน **P(next_class \| current_class)** จาก 2015→2025 (global trend) |
| **Cellular Automata** | แต่ละ pixel ดูเพื่อนบ้าน: ถ้าเพื่อนเป็นเมือง โอกาสเป็นเมืองเพิ่ม |

**Step = 10 ปี** (เพราะเรียนจาก 2015→2025 = 10 ปี)

> ⚠️ **Simplified:** งานวิจัยจริง CA-Markov ซับซ้อนกว่านี้ (มี suitability map, weights)<br>
> เราใช้เวอร์ชัน simplified เพื่อให้เข้าใจแนวคิดหลัก


In [ ]:
# ============ Step 1: Encode labels to contiguous integers ============
# LabelEncoder แปลง {0, 10, 20, ..., 80} → {0, 1, 2, ..., 7} เพื่อใช้เป็น index ของ matrix
# สำคัญ: ต้อง fit บน union ของ 2015 + 2025: เผื่อบาง class มีแค่ในช่วงใดช่วงหนึ่ง
encoder = LabelEncoder()
encoder.fit(np.concatenate([lulc_2015.flatten(), lulc_2025.flatten()]))
lulc_2015_enc = encoder.transform(lulc_2015.flatten()).reshape(lulc_2015.shape)
lulc_2025_enc = encoder.transform(lulc_2025.flatten()).reshape(lulc_2025.shape)
N_CLASSES = len(encoder.classes_)
print(f'Classes present: {[LULC_NAMES[c] for c in encoder.classes_]}')

# ============ Step 2: Build transition matrix ============
# P(i, j) = P(pixel class = j at t+1 | class = i at t)
transitions = confusion_matrix(
    lulc_2015_enc.flatten(),
    lulc_2025_enc.flatten(),
    normalize='true',  # แถวรวมเป็น 1
)

# ============ Visualize transition matrix ============
class_labels = [LULC_NAMES[c] for c in encoder.classes_]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(transitions, cmap='YlOrRd', vmin=0, vmax=1)

ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(class_labels, rotation=45, ha='right')
ax.set_yticklabels(class_labels)
ax.set_xlabel('To (2025)', fontweight='bold')
ax.set_ylabel('From (2015)', fontweight='bold')
ax.set_title('Transition Matrix: P(next | current)', fontsize=13, fontweight='bold')
ax.grid(False)

# Annotate each cell
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, f'{transitions[i, j]:.2f}',
                ha='center', va='center', fontsize=9,
                color='black' if transitions[i, j] < 0.5 else 'white')

plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
plt.tight_layout()
plt.show()

In [ ]:
def ca_markov_step(current_map, transitions, radius=1):
    """ทำ CA-Markov 1 step (= 10 ปี).

    สำหรับแต่ละ pixel:
        1. นับ class ของเพื่อนบ้านในกรอบ (2·radius+1) × (2·radius+1)
        2. influence = สัดส่วนของแต่ละ class ในเพื่อนบ้าน
        3. combined = transitions[current] × influence (element-wise)
        4. normalize แล้ว sample class ใหม่ตาม probability
    """
    n_classes = transitions.shape[0]
    H, W = current_map.shape
    next_map = np.copy(current_map)

    for i in tqdm(range(H), desc='rows', leave=False):
        for j in range(W):
            # Neighborhood bounds (with edge handling)
            y0, y1 = max(0, i - radius), min(H, i + radius + 1)
            x0, x1 = max(0, j - radius), min(W, j + radius + 1)
            neighbors = current_map[y0:y1, x0:x1].ravel()

            # Neighborhood influence (local)
            counts = np.bincount(neighbors, minlength=n_classes)
            influence = counts / counts.sum()

            # Combine with global transition (Markov)
            combined = transitions[current_map[i, j]] * influence
            total = combined.sum()
            if total == 0:
                continue  # keep current class
            combined = combined / total

            next_map[i, j] = np.random.choice(n_classes, p=combined)
    return next_map


# ============ Run simulation ============
N_STEPS = 2  # 2 steps × 10 ปี = 20 ปีข้างหน้า (2025 → 2045)

print(f'🔮 Simulating {N_STEPS} step(s) = {N_STEPS * 10} years into the future...')
state = lulc_2025_enc.copy()
for step in range(1, N_STEPS + 1):
    print(f'   Step {step}/{N_STEPS}...')
    state = ca_markov_step(state, transitions, radius=1)

# Decode กลับเป็น class ID เดิม
future_lulc = encoder.inverse_transform(state.flatten()).reshape(state.shape)
print('✓ Simulation complete')


In [ ]:
# Visualize 3 time periods
future_year = 2025 + N_STEPS * 10
fig, axes = plt.subplots(1, 3, figsize=(17, 6))

for ax, lulc, title in zip(
    axes,
    [lulc_2015, lulc_2025, future_lulc],
    [f'2015 (past)', f'2025 (present)', f'~{future_year} (simulated)'],
):
    ax.imshow(lulc, cmap=lulc_cmap, norm=lulc_norm, interpolation='nearest')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

legend_patches = [
    Patch(facecolor=LULC_COLORS_LIST[i], label=name)
    for i, name in enumerate(LULC_NAMES.values())
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4,
           fontsize=10, bbox_to_anchor=(0.5, -0.08), frameon=False)

fig.suptitle(f'LULC Evolution: Past → Present → Future', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Area comparison across 3 time periods
area_future = compute_area_by_class(future_lulc, PIXEL_AREA_M2)
area_future.columns = [f'~{future_year} (km²)']

combined = pd.concat([area_2015, area_2025, area_future], axis=1).fillna(0)
combined.columns = ['2015', '2025', f'~{future_year}']

combined['Δ 2015→2025'] = (combined['2025'] - combined['2015']).round(2)
combined[f'Δ 2025→{future_year}'] = (combined[f'~{future_year}'] - combined['2025']).round(2)
combined.round(2)


In [ ]:
# Timeline plot
fig, ax = plt.subplots(figsize=(11, 5.5))

years = [2015, 2025, future_year]
for class_name in combined.index:
    values = [combined.loc[class_name, '2015'],
              combined.loc[class_name, '2025'],
              combined.loc[class_name, f'~{future_year}']]
    # match color with LULC scheme
    try:
        cid = [k for k, v in LULC_NAMES.items() if v == class_name][0]
        color = LULC_COLORS_LIST[list(LULC_NAMES.keys()).index(cid)]
    except (IndexError, KeyError):
        color = None
    ax.plot(years, values, marker='o', markersize=9,
            label=class_name, color=color, linewidth=2.2)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Area (km²)', fontsize=12)
ax.set_title('LULC Area Timeline: Past / Present / Simulated',
             fontsize=13, fontweight='bold')
ax.set_xticks(years)
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 💰 10. Economic Interpretation + Applications

### สิ่งที่ทำได้จาก LULC maps

| Application | ประโยชน์ทางเศรษฐศาสตร์ |
|-------------|----------------------|
| **Urban sprawl tracking** | วางแผนโครงสร้างพื้นฐาน, ประเมินค่าที่ดิน |
| **Agricultural loss** | ประเมิน food security, รายได้เกษตรกร |
| **Green space mapping** | Hedonic pricing (ที่ดินใกล้สวน = ราคาสูง) |
| **Water body change** | Flood risk → insurance premium |
| **Bareland → Built-up** | ดัชนีการเติบโต (proxy GDP รายภูมิภาค) |

### ในโลกจริง

- 🏦 **World Bank / IMF** ใช้ **night lights** (VIIRS) เป็น proxy GDP ในพื้นที่ที่สถิติขาด (Henderson et al. 2012)
- 🏢 **Real estate firms** ใช้ LULC คาดคะเนราคาในโซนที่กำลังพัฒนา
- 🏛️ **GISTDA (ไทย)** ใช้ LULC วางแผนเมืองและภัยธรรมชาติ
- 🌍 **IPCC climate reports** ใช้ land cover change ประเมิน carbon emission

---

### 🧪 แบบฝึกหัด

เลือกทำ **อย่างน้อย 1 ข้อ:**

1. **เปลี่ยน AOI เป็นเมืองไทย** 🇹🇭
   - หา bbox จาก [bboxfinder.com](http://bboxfinder.com) (เช่น Bangkok, Chiang Mai)
   - เปลี่ยน `AOI_BBOX` แล้วรันใหม่
   - *หมายเหตุ:* pseudo-label thresholds ของเรา tune กับ BKK: เมืองอื่นอาจต้องปรับ NDBI threshold

2. **เปรียบเทียบหลายปี** 📅
   - เพิ่ม query ปี 2008, 2018 เข้ามา
   - Plot trend 4 จุดเวลาของแต่ละ class

3. **ปรับ RF Hyperparameters** 🎚️
   - ลอง `n_estimators` = 50, 200, 500
   - ลอง `max_depth` = 5, 10, None
   - เทียบ accuracy + เวลาเทรน

4. **Econ Analysis** 💼
   - Download GDP รายจังหวัดไทย (NESDC/สศช.)
   - คำนวณ correlation ระหว่าง Built-up growth vs GDP growth

5. **Simulate 5+ steps** 🔮
   - เปลี่ยน `N_STEPS = 5` → ไปปี 2074
   - Built-up จะขยายเป็นกี่เท่า?

---

### 💡 Capstone Project Ideas

- 🏙️ **Bangkok sprawl analysis**: ติดตาม 20 ปีย้อนหลัง, เทียบ GDP growth
- 🌾 **Crop yield prediction**: ใช้ NDVI timeseries ทำนายผลผลิต
- 🌊 **Flood risk mapping**: รวม LULC + DEM + ฝน → risk map
- 🏦 **Real estate intelligence**: ระบุโซน bareland → built-up (โอกาสลงทุน)
- 🌳 **Carbon stock estimation**: ใช้ LULC × carbon factor

**ส่งงาน:** Share Colab link ใน Google Classroom (3 สัปดาห์หลังสอบปลายภาค)

---

### 📚 แหล่งอ้างอิงสำหรับต่อยอด

- 🌏 [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/): Landsat, Sentinel, ฯลฯ
- 🛰️ [USGS EarthExplorer](https://earthexplorer.usgs.gov/): Landsat archive
- 🗺️ [GISTDA Data Portal](https://gi-portal.gistda.or.th/): ดาวเทียมไทย + ข้อมูลทางการ
- ⚡ [Google Earth Engine](https://earthengine.google.com/): cloud computing สำหรับ EO (advanced)

**Academic papers:**

- Henderson et al. (2012): *"Measuring Economic Growth from Outer Space"* (AER): night lights → GDP
- Donaldson & Storeygard (2016): *"The View from Above"* (JEP): satellite in econometrics
- Jensen (2007): *Remote Sensing of the Environment*: textbook พื้นฐาน
